In [9]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationSummaryBufferMemory
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    FewShotChatMessagePromptTemplate,
)


llm = ChatOpenAI(
    temperature=0.1,
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=70,
    return_messages=True,
)

examples = [
    {
        "question": "Top Gun",
        "answer": "🛩️👨‍✈️🔥",
    },
    {
        "question": "The Godfather",
        "answer": "👨‍👨‍👦🔫🍝",
    },
    {
        "question": "Titanic",
        "answer": "🚢💑🧊",
    },
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{question}"),
        ("ai", "{answer}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)


def load_memory(_):
    return memory.load_memory_variables({})["history"]


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "you are a helpful AI talking to a human",
        ),
        few_shot_prompt,
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    print(result.content)

In [10]:
invoke_chain("Interstellar")

🚀🌌⏳


In [11]:
invoke_chain("The Matrix")

💻🕶️🔵


In [12]:
invoke_chain("tell me all the movies that I asked.")

Sure! Here are the movies you asked about:

1. Top Gun
2. The Godfather
3. Titanic
4. Interstellar
5. The Matrix

I hope you enjoy watching them!
